# MuseTalk 1.5 — T4 Colab worker v3

**Clean replacement.** Uses an isolated Python 3.10 environment and explicitly installs OpenMIM before any `mim` command. No Hugging Face CLI is required.

Run one cell at a time. Do not rerun a failed cell blindly.

In [ ]:
# 1. Create the isolated Python 3.10 + CUDA/T4 environment
import sys, subprocess, shutil
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True, capture_output=True)
print("GPU:", gpu.stdout.strip() or "NONE")
if not gpu.stdout.strip():
    raise RuntimeError("STOP: No NVIDIA GPU. Select Runtime > Change runtime type > T4 GPU.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
MT = Path("/content/MuseTalk")
VENV = Path("/content/musetalk310")
if not MT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/TMElyralab/MuseTalk.git", str(MT)], check=True)
if VENV.exists() and not (VENV / "bin/python").exists():
    shutil.rmtree(VENV)
if not VENV.exists():
    subprocess.run(["uv", "venv", "--python", "3.10", "--seed", str(VENV)], check=True)
PY = str(VENV / "bin/python")
UV = ["uv", "pip", "install", "--python", PY]
subprocess.run(UV + ["pip==24.0", "setuptools==69.5.1", "wheel==0.43.0"], check=True)
subprocess.run(UV + ["torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2", "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
probe = subprocess.run([PY, "-c", "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"], text=True, capture_output=True, check=True)
print(probe.stdout)
if not torch_ok := (probe.stdout.strip().splitlines()[1:2] == ['True']):
    raise RuntimeError("STOP: CUDA is not visible inside Python 3.10.")
print("Environment OK.")


In [ ]:
# 2. Install MuseTalk dependencies — FIXED: install OpenMIM explicitly
import subprocess
from pathlib import Path
PY = "/content/musetalk310/bin/python"
MT = Path("/content/MuseTalk")
UV = ["uv", "pip", "install", "--python", PY]

# Base MuseTalk requirements.
subprocess.run(UV + ["-r", str(MT / "requirements.txt")], check=True)
# CRITICAL: the previous worker called /content/musetalk310/bin/mim without creating it.
subprocess.run(UV + ["openmim==0.3.9"], check=True)
MIM = "/content/musetalk310/bin/mim"
if not Path(MIM).exists():
    raise RuntimeError("OpenMIM installation failed: mim executable was not created.")

for pkg in ["mmengine", "mmcv==2.0.1", "mmdet==3.1.0", "mmpose==1.1.0"]:
    print("Installing", pkg)
    p = subprocess.run([MIM, "install", pkg], text=True, capture_output=True)
    print(p.stdout)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError("MMLab installation failed: " + pkg)

subprocess.run(UV + ["huggingface_hub==0.30.2", "gdown"], check=True)
check = subprocess.run([PY, "-c", "import torch, diffusers, transformers, mmcv, mmengine, mmdet, mmpose; print('IMPORTS_OK'); print(torch.cuda.is_available())"], text=True, capture_output=True)
print(check.stdout)
if check.returncode != 0:
    raise RuntimeError("Dependency import check failed:\n" + check.stderr)
if not check.stdout.strip().splitlines()[-1] == "True":
    raise RuntimeError("CUDA disappeared from the MuseTalk environment.")
print("MuseTalk dependencies are READY.")


In [ ]:
# 3. Download and verify every required model file
import subprocess
from pathlib import Path
PY = "/content/musetalk310/bin/python"
MODELS = Path("/content/MuseTalk/models")
MODELS.mkdir(parents=True, exist_ok=True)

download_code = r'''from huggingface_hub import snapshot_download
from pathlib import Path
def get(repo, dst, patterns):
    Path(dst).mkdir(parents=True, exist_ok=True)
    print('Downloading', repo)
    snapshot_download(repo_id=repo, local_dir=dst, allow_patterns=patterns)
get('TMElyralab/MuseTalk','/content/MuseTalk/models',['musetalkV15/unet.pth','musetalkV15/musetalk.json'])
get('stabilityai/sd-vae-ft-mse','/content/MuseTalk/models/sd-vae',['config.json','diffusion_pytorch_model.bin'])
get('openai/whisper-tiny','/content/MuseTalk/models/whisper',['config.json','preprocessor_config.json','pytorch_model.bin'])
get('yzd-v/DWPose','/content/MuseTalk/models/dwpose',['dw-ll_ucoco_384.pth'])'''
p = subprocess.run([PY, "-c", download_code], text=True)
if p.returncode: raise RuntimeError("Hugging Face model download failed.")

FACE = MODELS / "face-parse-bisent"
FACE.mkdir(parents=True, exist_ok=True)
face = FACE / "79999_iter.pth"
resnet = FACE / "resnet18-5c106cde.pth"
if not face.exists() or face.stat().st_size < 100000:
    subprocess.run(["/content/musetalk310/bin/gdown", "--id", "154JgKpzCPW82qINcVieuPH3fZ2e0P812", "-O", str(face)], check=True)
if not resnet.exists() or resnet.stat().st_size < 1000000:
    subprocess.run(["curl", "-L", "--fail", "--retry", "5", "-o", str(resnet), "https://download.pytorch.org/models/resnet18-5c106cde.pth"], check=True)

required = [MODELS/"musetalkV15/unet.pth", MODELS/"musetalkV15/musetalk.json", MODELS/"sd-vae/config.json", MODELS/"sd-vae/diffusion_pytorch_model.bin", MODELS/"whisper/config.json", MODELS/"whisper/preprocessor_config.json", MODELS/"whisper/pytorch_model.bin", MODELS/"dwpose/dw-ll_ucoco_384.pth", face, resnet]
for f in required:
    if not f.exists() or f.stat().st_size < 1000: raise RuntimeError(f"Missing/incomplete model: {f}")
print("ALL REQUIRED MODELS READY.")


In [ ]:
# 4. Upload the approved singer image/video and ACE-Step audio
from google.colab import files
from pathlib import Path
import shutil, subprocess
OUT = Path("/content/musetalk_output"); OUT.mkdir(parents=True, exist_ok=True)
print("Upload the APPROVED singer image/video:")
up = files.upload()
if not up: raise RuntimeError("No singer file uploaded.")
src = Path(next(iter(up))); singer = OUT / "singer_input" + src.suffix; shutil.move(str(src), str(singer))
print("Upload the successful ACE-Step bhajan MP3/WAV:")
up = files.upload()
if not up: raise RuntimeError("No audio file uploaded.")
src = Path(next(iter(up))); audio = OUT / "audio_input" + src.suffix; shutil.move(str(src), str(audio))
avatar = OUT / "avatar_source.png"; wav = OUT / "audio.wav"
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(singer), "-frames:v", "1", "-vf", "scale=512:-2", str(avatar)], check=True)
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(audio), "-ar", "16000", "-ac", "1", str(wav)], check=True)
print("Avatar:", avatar); print("Audio:", wav); print("INPUTS READY.")


In [ ]:
# 5. Run MuseTalk 1.5 and expose the final MP4
import subprocess, shutil
from pathlib import Path
MT = Path("/content/MuseTalk"); OUT = Path("/content/musetalk_output"); PY = "/content/musetalk310/bin/python"
cfg = MT / "configs/inference/colab_bhajan.yaml"
cfg.write_text(f'''bhajan_test:
  video_path: "{OUT / 'avatar_source.png'}"
  audio_path: "{OUT / 'audio.wav'}"
  bbox_shift: 0
''')
result_dir = OUT / "result"; result_dir.mkdir(parents=True, exist_ok=True)
cmd = [PY, "-m", "scripts.inference", "--inference_config", str(cfg), "--result_dir", str(result_dir), "--unet_model_path", str(MT/"models/musetalkV15/unet.pth"), "--unet_config", str(MT/"models/musetalkV15/musetalk.json"), "--version", "v15"]
print("STARTING MUSE TALK 1.5...")
p = subprocess.run(cmd, cwd=str(MT), text=True)
if p.returncode: raise RuntimeError(f"MuseTalk inference failed with exit code {p.returncode}.")
videos = list(result_dir.rglob("*.mp4"))
if not videos: raise RuntimeError("MuseTalk finished without producing an MP4.")
final = max(videos, key=lambda x: x.stat().st_size)
target = OUT / "bhajan_lipsync.mp4"; shutil.copy2(final, target)
print("SUCCESS:", target); print("SIZE MB:", round(target.stat().st_size/1024/1024, 2))
